# Prediction baseline research plan

**Scope:** Week-one evaluation of the community evolution prediction prototype.

**Current status:** The synthetic-data pipeline completed the table, temporal split, baseline prediction, and evaluation stages. The shared notebook still requires review and validation.

**Evidence run:** `lan-demo-final-20261020`

**Verified source revision:** `696cbb61da2faebb75f06a701afee943fe0199c0`

This notebook records the research design, evaluation evidence, limitations, and outstanding questions. Synthetic results must not be interpreted as real Reddit forecasting performance.

## Research question

Can the current prototype generate temporally valid predictions and evaluate community evolution events without using information unavailable at prediction time?

Week-one objectives:
- Verify the data-to-evaluation pipeline.
- Compare the available baseline models on the same temporal splits.
- Inspect class-level errors and potential leakage.
- Document what is still needed before evaluating real Reddit data.

The current experiment validates pipeline behavior on synthetic data. It does not establish real-world forecasting effectiveness or support conclusions about H3.

## Data and labels

The verified run uses synthetic community-month samples generated with seed 42.

- Total generated samples: 1,570.
- Labeled samples included in evaluation: 1,524.
- Unlabeled samples excluded from evaluation: 46.
- Event classes: stable, grow, shrink, merge, split, dissolve.

## Synthetic data and label generation

The current prototype uses synthetic community histories to test the forecasting pipeline before evaluating it on real Reddit data.

- **Dataset size:** 1,570 generated rows, including 1,524 labeled rows and 46 unlabeled rows.
- **Time span:** 36 months, from January 2015 to December 2017.
- **Random seed:** 42.
- **Target:** predict a community's next-month evolution label using information available up to the current month.
- **Feature timing:** features at time \(t\) must not use information from after time \(t\). The target corresponds to the subsequent month.
- **Unlabeled rows:** rows without a usable target label are not included in supervised evaluation.

The generator models community growth and evolution using synthetic histories, noise, and event-label rules. Its results are useful for testing the pipeline and evaluation logic, but they do not establish predictive performance on real Reddit communities.

Synthetic labels and features are not observations collected from Reddit. They must not be used to claim real-world performance.

## Split and leakage

The pipeline groups samples by label_month instead of randomly splitting community rows.

| Split | Samples | Labeled months |
|---|---:|---:|
| Train | 1,000 | 23 |
| Validation | 263 | 6 |
| Test | 261 | 6 |
| Unlabeled, excluded | 46 | 0 |

Required checks:
- Verify the exact first and last label_month in each split.
- Confirm that preprocessing and learned parameters use training data only.
- Verify that features are available at prediction time and targets refer to the intended future period.
- Run the negative-control leakage test and confirm it detects the deliberately leaky fixture.

These checks reduce known risks but do not prove that all possible leakage has been eliminated.

## Baselines and fairness

The verified evaluation includes:
- B1 Majority.
- B1 Persistence.
- B4 Logistic Regression.

All three models have metrics for train, validation, and test. Compare them on the same split and sample population.

## Baseline models and implementation

The prototype currently evaluates three baselines:

1. **B1 Majority:** predicts the most frequent class observed in the training data.
2. **B1 Persistence:** uses the previous evolution label as the prediction when that history is available.
3. **B4 Logistic Regression:** learns a classification model from community-level features.

For a fair comparison, models must use the same temporal train, validation, and test splits. Preprocessing and model fitting must use training data only. Validation and test data must not influence training or preprocessing decisions.

The implementation details—including the exact fallback behavior for missing history, feature list, model parameters, and preprocessing—should be checked against the source code before being reported as verified facts.

B2 Random Forest, B5 Gradient Boosting, HeTCo, and real Reddit features are not included in this verified run. No conclusion about their performance can be drawn.

## Results and evidence

The results cell below reads metrics and split counts from the verified run's artifacts rather than manually entering scores.

Primary metrics:
- macro_f1: F1 averaged over classes present in the true labels.
- macro_f1_six: F1 averaged over all six event classes.
- balanced_accuracy: mean recall over classes present in the true labels.

All six classes are present in the current train, validation, and test splits, so macro_f1 and macro_f1_six coincide in this run.

Interpret results alongside per-class metrics and confusion matrices. Report classes with no correct detections, not only aggregate scores.

All reported metrics must be traceable to the saved run artifacts. If the artifact, configuration, or source revision differs from the verified run, rerun the checks before treating the results as reproducible.

## Critical review

In the test split, B1 Persistence achieves macro-F1 of approximately 0.2557, B4 Logistic Regression approximately 0.2160, and B1 Majority approximately 0.1125.

Persistence performs best among the three currently evaluated models, but the scores remain limited. The confusion matrices show that some event classes are not detected correctly. Aggregate metrics alone therefore provide an incomplete picture.

The test period contains only six labeled months, and the experiment uses synthetic data. These results do not establish performance on real Reddit communities. Further review is needed for temporal boundaries, feature availability, class-level errors, and leakage controls.

## Implementation

Run the pipeline from the repository root using the project's virtual environment and explicit configuration:

```powershell
& .\.venv\Scripts\python.exe -m reddit_sna.pipelines.prediction `
  --config .\configs\prediction.toml `
  --run-id <unique-run-id>
```

For the verified run, inspect its manifest, split_summary.csv, metrics.csv, per_class.csv, and confusion matrices under artifacts/prediction/lan-demo-final-20261020/.

Do not modify sys.path to make imports work. Keep reusable production logic in src/reddit_sna rather than duplicating it in the notebook.

In [1]:

import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import display


# --------------------------------------------------
# 1. Locate the repository root without modifying sys.path
# --------------------------------------------------

REPO_CONFIG = Path("configs/prediction.toml")
REPO_SOURCE = Path("src/reddit_sna")

RUN_CANDIDATES = (
    "lan-demo-final-20261020",
)

REQUIRED_FILES = [
    "manifest.json",
    "split_summary.csv",
    "metrics.csv",
    "per_class.csv",
    "confusion_B1_majority_test.csv",
    "confusion_B1_persistence_test.csv",
    "confusion_B4_logreg_test.csv",
]


def is_repo_root(path):
    return (
        (path / REPO_CONFIG).is_file()
        and (path / REPO_SOURCE).is_dir()
    )


def find_repo_roots(start):
    """Search ancestors first, then descendants up to four levels deep."""
    found = []
    seen = set()

    # First check the current directory and its parents.
    for path in (start, *start.parents):
        path = path.resolve()
        if is_repo_root(path) and path not in seen:
            found.append(path)
            seen.add(path)

    # If necessary, search below the current working directory.
    # Skip folders that are large, generated, or irrelevant.
    excluded_dirs = {
        ".git",
        ".venv",
        "venv",
        "node_modules",
        "__pycache__",
        ".ipynb_checkpoints",
        "site-packages",
        "AppData",
    }

    max_depth = 4

    for dirpath, dirnames, _ in os.walk(start, topdown=True):
        current_path = Path(dirpath).resolve()

        try:
            depth = len(current_path.relative_to(start).parts)
        except ValueError:
            dirnames[:] = []
            continue

        dirnames[:] = [
            name for name in dirnames
            if name not in excluded_dirs
        ]

        if depth >= max_depth:
            dirnames[:] = []

        if is_repo_root(current_path) and current_path not in seen:
            found.append(current_path)
            seen.add(current_path)

    return found


current = Path.cwd().resolve()
repo_roots = find_repo_roots(current)

if not repo_roots:
    raise RuntimeError(
        f"Repository root not found from {current}. "
        "Open the notebook within the project workspace, "
        "or set the notebook's working directory to the project."
    )


# --------------------------------------------------
# 2. Find runs that contain all required evidence files
# --------------------------------------------------

def complete_runs(repo_root):
    """Return candidate runs with every required artifact present."""
    results = []

    for candidate in RUN_CANDIDATES:
        candidate_dir = (
            repo_root / "artifacts" / "prediction" / candidate
        )

        if (
            candidate_dir.is_dir()
            and all(
                (candidate_dir / filename).is_file()
                for filename in REQUIRED_FILES
            )
        ):
            results.append((candidate, candidate_dir))

    return results


valid_repositories = []

for root in repo_roots:
    runs = complete_runs(root)
    if runs:
        valid_repositories.append((root, runs))


if not valid_repositories:
    searched = "\n".join(f"- {root}" for root in repo_roots)
    raise FileNotFoundError(
        "No repository was found with a complete supported evidence run.\n"
        f"Searched repositories:\n{searched}\n"
        "Expected run IDs: "
        + ", ".join(RUN_CANDIDATES)
        + "\nCheck that the artifacts are present or run the pipeline."
    )


# Avoid silently selecting the wrong project if multiple copies exist.
if len(valid_repositories) > 1:
    options = "\n".join(
        f"- {root}" for root, _ in valid_repositories
    )
    raise RuntimeError(
        "Multiple repositories contain complete evidence runs. "
        "Set the notebook working directory to the intended project "
        "and rerun this cell:\n"
        f"{options}"
    )


repo_root, available_runs = valid_repositories[0]

# Select the single evidence run documented in this notebook.
run_id, run_dir = next(
    (run_id, path)
    for candidate in RUN_CANDIDATES
    for run_id, path in available_runs
    if run_id == candidate
)


# --------------------------------------------------
# 3. Validate the run manifest and pipeline stages
# --------------------------------------------------

manifest = json.loads(
    (run_dir / "manifest.json").read_text(encoding="utf-8")
)

assert manifest.get("status") == "complete_through_baselines", (
    f"Unexpected run status: {manifest.get('status')}"
)

stage_status = manifest.get("stage_status", {})
required_stages = ("table", "split", "baselines", "evaluation")

assert all(
    stage_status.get(stage) == "complete"
    for stage in required_stages
), f"Incomplete pipeline stages: {stage_status}"

print("Repository:", repo_root)
print("Run:", run_id)
print("Evidence directory:", run_dir)
print("Source revision:", manifest.get("git", {}).get("revision"))
print("Working tree dirty:", manifest.get("git", {}).get("dirty"))
print("Status:", manifest["status"])
print("Data kind:", manifest.get("data_kind"))


# --------------------------------------------------
# 4. Load and validate split summary and aggregate metrics
# --------------------------------------------------

split_summary = pd.read_csv(run_dir / "split_summary.csv")
metrics = pd.read_csv(run_dir / "metrics.csv")

expected_counts = {
    "train": 1000,
    "validation": 263,
    "test": 261,
    "unlabeled_dropped": 46,
}

assert {"split", "n_samples", "n_label_months"}.issubset(
    split_summary.columns
), f"Unexpected split schema: {split_summary.columns.tolist()}"

assert split_summary["split"].is_unique, (
    "Duplicate split names in split_summary.csv"
)

observed_counts = (
    split_summary.set_index("split")["n_samples"].to_dict()
)

assert observed_counts == expected_counts, (
    f"Unexpected sample counts: {observed_counts}"
)

assert int(split_summary["n_samples"].sum()) == 1570


expected_models = {
    "B1_majority",
    "B1_persistence",
    "B4_logreg",
}

expected_splits = {"train", "validation", "test"}

assert len(metrics) == 9, (
    f"Expected 9 metric rows, got {len(metrics)}"
)
assert set(metrics["model"]) == expected_models
assert set(metrics["split"]) == expected_splits
assert not metrics.duplicated(["model", "split"]).any()

expected_pairs = {
    (model, split)
    for model in expected_models
    for split in expected_splits
}

observed_pairs = set(zip(metrics["model"], metrics["split"]))

assert observed_pairs == expected_pairs

metric_columns = [
    "macro_f1",
    "macro_f1_six",
    "balanced_accuracy",
]

assert set(metric_columns).issubset(metrics.columns), (
    f"Missing metric columns: {metric_columns}"
)

for column in metric_columns:
    assert metrics[column].notna().all(), (
        f"Missing values in {column}"
    )
    assert metrics[column].between(0, 1).all(), (
        f"Out-of-range values in {column}"
    )

print("PASS: artifact consistency checks completed.")


# --------------------------------------------------
# 5. Display split summary and aggregate metrics
# --------------------------------------------------

print("\nSplit summary")
display(split_summary)

print("\nAggregate evaluation metrics")
display(
    metrics[
        [
            "model",
            "split",
            "n",
            "n_classes_present",
            "macro_f1",
            "macro_f1_six",
            "balanced_accuracy",
        ]
    ].sort_values(["split", "model"])
)


# --------------------------------------------------
# 6. Display per-class metrics on the test split
# --------------------------------------------------

per_class = pd.read_csv(run_dir / "per_class.csv")

assert {"split", "model", "label"}.issubset(per_class.columns), (
    f"Unexpected per_class.csv schema: {per_class.columns.tolist()}"
)

per_class_test = (
    per_class[per_class["split"] == "test"]
    .sort_values(["model", "label"])
)

print("\nPer-class evaluation metrics — TEST")
display(per_class_test)

# Highlight classes with zero F1, if the column is available.
if "f1" in per_class_test.columns:
    zero_f1 = per_class_test[per_class_test["f1"] == 0]

    print("\nTest classes with F1 = 0")
    if zero_f1.empty:
        print("No zero-F1 rows found.")
    else:
        display(zero_f1)


# --------------------------------------------------
# 7. Display test confusion matrices
# --------------------------------------------------

confusion_files = [
    "confusion_B1_majority_test.csv",
    "confusion_B1_persistence_test.csv",
    "confusion_B4_logreg_test.csv",
]

for filename in confusion_files:
    confusion_path = run_dir / filename
    confusion_df = pd.read_csv(confusion_path)

    print(f"\nTest confusion matrix: {filename}")
    display(confusion_df)

Repository: <PROJECT_ROOT>
Run: lan-demo-final-20261020
Evidence directory: <PROJECT_ROOT>\artifacts\prediction\lan-demo-final-20261020
Source revision: facbc591ed2efe08bed57f4fbe00e1d4dee7d550
Working tree dirty: False
Status: complete_through_baselines
Data kind: synthetic
PASS: artifact consistency checks completed.

Split summary


,split,n_samples,n_label_months,n_stable,n_grow,n_shrink,n_merge,n_split,n_dissolve
0,train,1000,23,500,165,160,65,52,58
1,validation,263,6,142,42,35,16,11,17
2,test,261,6,133,46,37,22,9,14
3,unlabeled_dropped,46,0,0,0,0,0,0,0



Aggregate evaluation metrics


,model,split,n,n_classes_present,macro_f1,macro_f1_six,balanced_accuracy
0,B1_majority,test,261,6,0.112521,0.112521,0.166667
3,B1_persistence,test,261,6,0.255693,0.255693,0.258584
6,B4_logreg,test,261,6,0.215966,0.215966,0.215521
1,B1_majority,train,1000,6,0.111111,0.111111,0.166667
4,B1_persistence,train,1000,6,0.279076,0.279076,0.284249
7,B4_logreg,train,1000,6,0.261774,0.261774,0.281974
2,B1_majority,validation,263,6,0.116872,0.116872,0.166667
5,B1_persistence,validation,263,6,0.257502,0.257502,0.264336
8,B4_logreg,validation,263,6,0.229113,0.229113,0.232402



Per-class evaluation metrics — TEST


...

,y_true,stable,grow,shrink,merge,split,dissolve
0,stable,133,0,0,0,0,0
1,grow,46,0,0,0,0,0
2,shrink,37,0,0,0,0,0
3,merge,22,0,0,0,0,0
4,split,9,0,0,0,0,0
5,dissolve,14,0,0,0,0,0



Test confusion matrix: confusion_B1_persistence_test.csv


,y_true,stable,grow,shrink,merge,split,dissolve
0,stable,89,13,12,14,5,0
1,grow,21,19,0,4,2,0
2,shrink,21,0,14,2,0,0
3,merge,9,6,4,2,1,0
4,split,3,4,1,1,0,0
5,dissolve,9,0,4,0,1,0



Test confusion matrix: confusion_B4_logreg_test.csv


,y_true,stable,grow,shrink,merge,split,dissolve
0,stable,70,17,24,16,1,5
1,grow,19,14,5,4,1,3
2,shrink,20,0,13,4,0,0
3,merge,9,6,6,0,1,0
4,split,3,3,2,0,1,0
5,dissolve,5,1,6,2,0,0


## Handoff and limitations

Verified source revision: 696cbb61da2faebb75f06a701afee943fe0199c0.

The verified manifest records a clean working tree and successful completion of the table, split, baseline, and evaluation stages. The experiment is synthetic and covers only the three available baselines.

Outstanding work:
- Cross-check the synthetic generator rules, label timing, baseline fallback behavior, feature list, and preprocessing against the source implementation.
- Review per-class metrics and test confusion matrices, especially event classes with zero recall.
- Verify temporal boundaries, feature-time constraints, and the negative-control leakage test.
- Complete the remaining planned models: B2 Random Forest, B5 Gradient Boosting, and HeTCo.
- Evaluate on suitable real Reddit data when available.
- Cross-review evaluation outputs and document unresolved limitations.

The current evidence run uses synthetic data and the source revision recorded in its manifest. The results validate the current pipeline stages but do not establish real-world Reddit forecasting performance.

Before handoff, restart the kernel and run all cells. Save an executed local copy for review, then clear all outputs in the shared notebook before committing it. Do not claim that opening or editing the template alone completes an experiment.